# LoCoMo eval on a Kaggle GPU

Runs the LLM stages of [`evals/locomo`](https://github.com/dfavelava/connectome/tree/main/evals/locomo) on a free Kaggle GPU:

- **`extract`** - `locomo-eval extract`: a local model reads each conversation and chooses the memories to store. `EXTRACT_PROMPT` picks the add-only (`extract_v1`, `extract_v2`) or lifecycle (`lifecycle_v1`) variant; run them with the same model and samples to compare them.
- **`answer`** - `locomo-eval answer`: answers and judges a retrieval run you made on your own machine.

Neither stage talks to the Connectome backend, so all this notebook needs is Ollama and the harness.

**One-time setup**

1. Verify your phone number on Kaggle (needed for GPUs and internet).
2. Create a **private** Kaggle Dataset (LoCoMo is CC BY-NC, so don't publish it) holding `locomo10.json`, and for the `answer` stage the retrieval run's `results/<run-id>.json`.
3. Import this notebook (File -> Import Notebook), then in the settings panel set **Accelerator: GPU T4 x2** (or P100), **Internet: on**, and add your dataset as an input.
4. If the repository is private, add a GitHub token as a Kaggle secret named `GITHUB_TOKEN` (Add-ons -> Secrets) and attach it.

**Each session**

1. Edit the configuration cell below.
2. **Save Version -> Save & Run All (Commit)**. The run carries on in the background with the browser closed, for up to about 12 hours.
3. Download `results/` from the version's **Output** tab.

**Resuming.** Everything the harness finishes is cached in `results/`, one fsynced line per call. To pick up where a version stopped, add that version's output as an input (Add Input -> Your Work -> this notebook) and commit again: the restore cell finds every attached `results/` folder, and the stage skips what is already done. Start with `SAMPLES = "conv-26"` and multiply its time by ~13 to estimate a full run; GPU time counts against Kaggle's weekly quota.


In [ ]:
# --- Configuration: edit, then Save Version -> Save & Run All (Commit) ---

# "extract" (memory extraction) or "answer" (answer + judge a retrieval run).
STAGE = "extract"

# Ollama model tags, pulled at the start of every session.
EXTRACTOR_MODEL = "qwen3:8b"
# For STAGE = "extract": "extract_v1" or "extract_v2" (add-only) or
# "lifecycle_v1" (recalls earlier memories, skips duplicates and supersedes
# stale ones). All are cached side by side in results/extractions.jsonl.
EXTRACT_PROMPT = "extract_v1"
ANSWER_MODEL = "qwen3:8b"
JUDGE_MODEL = "qwen3:14b"

# For STAGE = "answer": the retrieval run to score. <RUN_ID>.json must be in an
# attached dataset or a previous version's output.
RUN_ID = "base"

# Comma-separated sample ids ("conv-26" for a quick timing run), or None for all.
SAMPLES = "conv-26"

# The stage is stopped after this long, leaving time to save the output before
# Kaggle's ~12 hour limit. Finished work is kept; committing again resumes it.
TIME_LIMIT_HOURS = 11.0

# LLM calls in flight (and OLLAMA_NUM_PARALLEL). 2 keeps both GPUs of a
# T4 x2 busy; use 1 on a P100.
CONCURRENCY = 2

# The connectome branch or tag to run.
REPO_URL = "https://github.com/dfavelava/connectome.git"
REPO_REF = "main"


## Setup

Paths, a helper that streams command output, and a GPU check.

In [ ]:
import base64
import json
import os
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

if STAGE not in ("extract", "answer"):
    raise ValueError(f"STAGE must be 'extract' or 'answer', not {STAGE!r}")

# Kaggle's paths; the environment overrides only exist for testing the notebook elsewhere.
INPUT_DIR = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))
WORK_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
# Only /kaggle/working is saved as the version's output, so the code, models
# and caches live elsewhere and results/ is all that gets saved.
SCRATCH_DIR = Path(os.environ.get("LOCOMO_SCRATCH_DIR", "/tmp/locomo"))
RESULTS_DIR = WORK_DIR / "results"
REPO_DIR = SCRATCH_DIR / "connectome"
EVAL_DIR = REPO_DIR / "evals" / "locomo"
OLLAMA_URL = "http://127.0.0.1:11434"

os.environ["PATH"] = f"{Path.home() / '.local' / 'bin'}:{os.environ['PATH']}"
os.environ["OLLAMA_MODELS"] = str(SCRATCH_DIR / "ollama-models")
os.environ["OLLAMA_NUM_PARALLEL"] = str(CONCURRENCY)
os.environ["OLLAMA_HOST"] = OLLAMA_URL
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)


def sh(*args, check=True, cwd=None, env=None):
    """Run a command, streaming its output into the cell."""
    print("$", " ".join(map(str, args)), flush=True)
    with subprocess.Popen(list(map(str, args)), cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    if check and proc.returncode:
        raise subprocess.CalledProcessError(proc.returncode, args)
    return proc.returncode


if shutil.which("nvidia-smi"):
    sh("nvidia-smi", check=False)
else:
    print("WARNING: no GPU found; set Accelerator to GPU in the notebook settings, or this will be very slow.")


## Install Ollama, uv and the harness

In [ ]:
# Ollama and uv; uv installs the Python version the harness needs by itself.
if not shutil.which("ollama"):
    if not shutil.which("zstd") and shutil.which("apt-get"):
        # Newer Ollama releases ship as .tar.zst.
        sh("bash", "-c", "apt-get update -qq && apt-get install -y -qq zstd", check=False)
    sh("bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh")
if not shutil.which("uv"):
    sh("bash", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh")

# The token goes in an HTTP header through git's environment config, so it never
# appears in a command line or in the output.
git_env = dict(os.environ)
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")

if not REPO_DIR.exists():
    sh("git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO_DIR, env=git_env)
sh("git", "-C", REPO_DIR, "log", "-1", "--format=running connectome %H (%s)")
sh("uv", "sync", "--quiet", cwd=EVAL_DIR)


## Start Ollama and pull the models

Models are kept outside `/kaggle/working`, so they aren't saved with the output.

In [ ]:
def ollama_up():
    try:
        urllib.request.urlopen(f"{OLLAMA_URL}/api/version", timeout=2)
        return True
    except OSError:
        return False


if not ollama_up():
    log = open(SCRATCH_DIR / "ollama.log", "a")
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(120):
        if ollama_up():
            break
        time.sleep(1)
    else:
        raise RuntimeError(f"Ollama did not start; see {SCRATCH_DIR / 'ollama.log'}")

models = [EXTRACTOR_MODEL] if STAGE == "extract" else [ANSWER_MODEL, JUDGE_MODEL]
for model in dict.fromkeys(models):
    sh("ollama", "pull", model)


## Restore finished work

In [ ]:
# Bring back finished work: every results/ folder among the inputs (earlier
# versions' outputs), then loose run files from uploaded datasets. Cache files
# (.jsonl) are merged line by line - the harness keys each line, so the union
# is safe - and other files are copied unless already present.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def restore_file(src):
    dest = RESULTS_DIR / src.name
    if src.suffix == ".jsonl":
        seen = set(dest.read_text(encoding="utf-8").splitlines()) if dest.exists() else set()
        added = 0
        with dest.open("a", encoding="utf-8") as f:
            for line in src.read_text(encoding="utf-8").splitlines():
                if line.strip() and line not in seen:
                    f.write(line + "\n")
                    seen.add(line)
                    added += 1
        print(f"merged {added} lines from {src}")
    elif not dest.exists():
        shutil.copy2(src, dest)
        print(f"copied {src}")


previous_outputs = sorted(p for p in INPUT_DIR.rglob("results") if p.is_dir())
for folder in previous_outputs:
    for src in sorted(folder.iterdir()):
        if src.is_file() and src.suffix in (".json", ".jsonl"):
            restore_file(src)
for src in sorted(INPUT_DIR.rglob("*.json*")):
    if src.is_file() and src.suffix in (".json", ".jsonl") and src.name != "locomo10.json" and not any(folder in src.parents for folder in previous_outputs):
        restore_file(src)

data_path = next(iter(sorted(INPUT_DIR.rglob("locomo10.json"))), None)
if STAGE == "extract" and data_path is None:
    raise FileNotFoundError(f"no locomo10.json under {INPUT_DIR}; add your private LoCoMo dataset as an input")
if STAGE == "answer" and not (RESULTS_DIR / f"{RUN_ID}.json").exists():
    raise FileNotFoundError(f"no {RUN_ID}.json among the inputs; upload results/{RUN_ID}.json from your retrieval run to your dataset")
print(f"results in {RESULTS_DIR}: {sorted(p.name for p in RESULTS_DIR.iterdir())}")


## Run the stage

In [ ]:
if STAGE == "extract":
    command = [
        "locomo-eval", "extract",
        "--data", data_path,
        "--cache", RESULTS_DIR / "extractions.jsonl",
        "--extractor-model", f"ollama:{EXTRACTOR_MODEL}",
        "--extract-prompt", EXTRACT_PROMPT,
        "--concurrency", CONCURRENCY,
    ]
else:
    command = [
        "locomo-eval", "answer", RUN_ID,
        "--results-dir", RESULTS_DIR,
        "--answer-model", f"ollama:{ANSWER_MODEL}",
        "--judge-model", f"ollama:{JUDGE_MODEL}",
        "--concurrency", CONCURRENCY,
    ]
if SAMPLES:
    command += ["--samples", SAMPLES]

# `timeout` stops the stage before Kaggle's limit; a nonzero exit is reported
# rather than raised, so the version still finishes and saves its output.
started = time.monotonic()
code = sh("timeout", "-k", "60", int(TIME_LIMIT_HOURS * 3600), "uv", "run", "--quiet", *command, check=False, cwd=EVAL_DIR)
elapsed = (time.monotonic() - started) / 3600
if code == 0:
    print(f"\n{STAGE} finished in {elapsed:.2f} h")
elif code in (124, 137):
    print(f"\n{STAGE} stopped at the {TIME_LIMIT_HOURS:g} h limit; add this version's output as an input and commit again to resume")
else:
    print(f"\n{STAGE} exited with code {code} after {elapsed:.2f} h; see the log above. Finished work is saved; committing again resumes it")


## Summary

In [ ]:
for path in sorted(RESULTS_DIR.iterdir()):
    print(f"{path.stat().st_size / 1e6:9.2f} MB  {path.name}")

cache = RESULTS_DIR / "extractions.jsonl"
if STAGE == "extract" and cache.exists():
    ok, failed = {}, {}
    for line in cache.read_text(encoding="utf-8").splitlines():
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue
        if record.get("extractor_model") != f"ollama:{EXTRACTOR_MODEL}" or record.get("prompt_version") != EXTRACT_PROMPT:
            continue
        target = ok if record["status"] == "ok" else failed
        target.setdefault(record["sample_id"], set()).add(record["session"])
    for sample in sorted(ok.keys() | failed.keys()):
        still_failed = failed.get(sample, set()) - ok.get(sample, set())
        print(f"{sample}: {len(ok.get(sample, ()))} sessions extracted with {EXTRACT_PROMPT}" + (f", failed: {sorted(still_failed)}" if still_failed else ""))
print("\nDownload results/ from this version's Output tab.")
